# Compositional Structure Discovery

Automated discovery of compositional algorithmic structure: information
bottlenecks, reusable subroutines (head clusters), skip connection roles,
algorithmic decomposition, and layer group analysis.

This notebook covers the `irtk.compositional_structure` module.

## Why This Matters

Compositional structure analysis identifies how the model decomposes complex computations into modular subroutines. Information bottleneck analysis, subroutine clustering, and algorithmic decomposition reveal the model's computational architecture at a higher level than individual heads.

**Key references:**
- [Conmy et al. (2023) "Towards Automated Circuit Discovery"](https://arxiv.org/abs/2304.14997) — Automated methods for circuit finding via edge pruning
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import compositional_structure

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Information Bottleneck Layers

Where does the model compress representations?

In [ ]:
prompts = [
    "The Eiffel Tower is located in",
    "The capital of France is",
    "London is the capital of",
    "Berlin is a city in",
]
token_seqs = [model.to_tokens(p) for p in prompts]

result = compositional_structure.information_bottleneck_layers(model, token_seqs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(result['effective_dims'], 'bo-')
axes[0].set_xlabel('Component (0=embed, 1+=layer)')
axes[0].set_ylabel('Effective Dimensionality')
axes[0].set_title('Information Bottleneck Analysis')
for bl in result['bottleneck_layers']:
    axes[0].axvline(bl, color='red', alpha=0.3)

axes[1].bar(range(len(result['compression_ratios'])), result['compression_ratios'])
axes[1].axhline(1.0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Layer')
axes[1].set_ylabel('Compression Ratio (next/prev)')
axes[1].set_title('Per-Layer Compression')

plt.tight_layout()
plt.show()
print(f"Tightest bottleneck: layer {result['tightest_bottleneck']}")

## 2. Subroutine Clustering

Which attention heads compute similar functions?

In [ ]:
result = compositional_structure.subroutine_clustering(model, token_seqs, n_clusters=6)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Similarity matrix
im = axes[0].imshow(result['similarity_matrix'], cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
axes[0].set_title('Head Similarity Matrix')
axes[0].set_xlabel('Head Index')
axes[0].set_ylabel('Head Index')
plt.colorbar(im, ax=axes[0])

# Cluster assignments as a grid
assignments = result['cluster_assignments'].reshape(model.cfg.n_layers, model.cfg.n_heads)
im = axes[1].imshow(assignments, cmap='Set3', aspect='auto')
axes[1].set_xlabel('Head')
axes[1].set_ylabel('Layer')
axes[1].set_title('Head Cluster Assignments')
plt.colorbar(im, ax=axes[1], label='Cluster ID')

plt.tight_layout()
plt.show()

for c in range(6):
    members = [result['head_labels'][i] for i in range(len(result['cluster_assignments']))
               if result['cluster_assignments'][i] == c]
    if members:
        print(f"Cluster {c}: {', '.join(members[:8])}{'...' if len(members) > 8 else ''}")

## 3. Algorithmic Decomposition

Decompose computation into sequential phases.

In [ ]:
tokens = model.to_tokens("The Eiffel Tower is located in")
paris_id = tokenizer.encode(" Paris")[0]
def metric(logits):
    return float(logits[-1, paris_id])

result = compositional_structure.algorithmic_decomposition(model, tokens, metric)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(result['cumulative_metric'], 'bo-')
for b in result['phase_boundaries'][1:-1]:
    ax.axvline(b, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Layers included (0=embed only, N=full model)')
ax.set_ylabel('Metric (Paris logit)')
ax.set_title(f'Algorithmic Decomposition ({result["n_phases"]} phases)')
plt.tight_layout()
plt.show()

print(f"Phase boundaries: {result['phase_boundaries']}")
for i, contrib in enumerate(result['phase_contributions']):
    start, end = result['phase_boundaries'][i], result['phase_boundaries'][i+1]
    print(f"  Phase {i} (layers {start}-{end-1}): contribution = {contrib:+.4f}")

## 4. Layer Group Analysis

Which layer groups implement which capabilities?

In [ ]:
result = compositional_structure.generalization_phase_analysis(
    model, token_seqs, metric
)

fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(result['group_labels']))
ax.bar(x, result['group_effects'], yerr=result['group_std'],
       color='steelblue', capsize=5)
ax.set_xticks(x)
ax.set_xticklabels(result['group_labels'])
ax.set_ylabel('Effect when ablated')
ax.set_title('Layer Group Importance')
plt.tight_layout()
plt.show()

print(f"Most important group: {result['group_labels'][result['most_important_group']]}")

## Summary

| Function | Purpose |
|----------|--------|
| `information_bottleneck_layers()` | Find information compression layers |
| `subroutine_clustering()` | Cluster heads by attention pattern similarity |
| `skip_connection_importance()` | Measure skip vs computation importance |
| `algorithmic_decomposition()` | Decompose into sequential computation phases |
| `generalization_phase_analysis()` | Layer group capability analysis |